# 🖥️ Dashboard — Đồ án Học máy SV16
## So sánh dự báo Flow giữa hai tập Sensor trong PeMSD3

**Chạy file này SAU khi đã chạy `main.ipynb` để có mô hình + dữ liệu.**

---

In [1]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
from pathlib import Path

from modules import data_loader, feature_engineering as fe, models, visualization as viz

%matplotlib inline
print('✅ Thư viện đã sẵn sàng')

✅ Thư viện đã sẵn sàng


## 1. Load dữ liệu đã xử lý

In [2]:
CLEAN_DIR = '__datasets-clean'
IMAGES_DIR = 'resultImages'
MODELS_DIR = 'models'

# Load tất cả datasets
data = {}
for name in ['baselineA_train', 'baselineA_valid', 'baselineA_test',
             'baselineB_train', 'baselineB_valid', 'baselineB_test']:
    path = f'{CLEAN_DIR}/{name}.csv'
    if Path(path).exists():
        data[name] = pd.read_csv(path)
        data[name]['timestamp'] = pd.to_datetime(data[name]['timestamp'])
        print(f'  ✅ {name}: {len(data[name]):,} dòng')
    else:
        print(f'  ❌ {name}: không tìm thấy → chạy main.ipynb trước!')

# Load raw data cho EDA
df_raw = data_loader.load_raw_data('__datasets-raw/SV16_PeMSD3_sample_8sensors.csv')

  ✅ baselineA_train: 16,916 dòng
  ✅ baselineA_valid: 3,624 dòng
  ✅ baselineA_test: 3,628 dòng
  ✅ baselineB_train: 16,916 dòng
  ✅ baselineB_valid: 3,624 dòng
  ✅ baselineB_test: 3,628 dòng
✅ Đã load 48,384 dòng từ SV16_PeMSD3_sample_8sensors.csv
   Sensors: ['PEMSD3_007', 'PEMSD3_008', 'PEMSD3_009', 'PEMSD3_010', 'PEMSD3_011', 'PEMSD3_012', 'PEMSD3_013', 'PEMSD3_014']
   Thời gian: 2024-02-01 00:00:00 → 2024-02-21 23:55:00


## 2. Load mô hình đã train

In [3]:
import joblib, glob

saved_models = {'Baseline_A': {}, 'Baseline_B': {}}
for baseline in ['Baseline_A', 'Baseline_B']:
    model_dir = Path(MODELS_DIR) / baseline
    if model_dir.exists():
        for f in sorted(model_dir.glob('*.pkl')):
            name = f.stem
            saved_models[baseline][name] = joblib.load(f)
            print(f'  ✅ Loaded {baseline}/{name}')
    else:
        print(f'  ❌ {baseline}: không tìm thấy mô hình')

print(f'\nTổng: Baseline A = {len(saved_models["Baseline_A"])} models, '
      f'Baseline B = {len(saved_models["Baseline_B"])} models')

  ✅ Loaded Baseline_A/1_LinearRegression
  ✅ Loaded Baseline_A/2_Ridge
  ✅ Loaded Baseline_A/3_KNN
  ✅ Loaded Baseline_A/4_DecisionTree
  ✅ Loaded Baseline_A/5_RandomForest
  ✅ Loaded Baseline_A/6_XGBoost
  ✅ Loaded Baseline_B/1_LinearRegression
  ✅ Loaded Baseline_B/2_Ridge
  ✅ Loaded Baseline_B/3_KNN
  ✅ Loaded Baseline_B/4_DecisionTree
  ✅ Loaded Baseline_B/5_RandomForest
  ✅ Loaded Baseline_B/6_XGBoost

Tổng: Baseline A = 6 models, Baseline B = 6 models


## 3. Dashboard — Chọn sensor & xem dữ liệu gốc

In [4]:
all_sensors = sorted(df_raw['sensor_id'].unique())

sensor_dropdown = widgets.Dropdown(
    options=all_sensors,
    value=all_sensors[0],
    description='Sensor:',
)

metric_dropdown = widgets.Dropdown(
    options=['flow', 'speed', 'occupancy'],
    value='flow',
    description='Metric:',
)

out_raw = widgets.Output()

def update_raw_plot(change):
    with out_raw:
        clear_output(wait=True)
        sid = sensor_dropdown.value
        col = metric_dropdown.value
        subset = df_raw[df_raw['sensor_id'] == sid].sort_values('timestamp')
        
        fig, ax = plt.subplots(figsize=(16, 4))
        ax.plot(subset['timestamp'], subset[col], linewidth=0.7, alpha=0.9)
        ax.set_title(f'{col.upper()} — {sid}', fontsize=14)
        ax.set_ylabel(col)
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()
        
        # Thống kê nhanh
        print(f'📊 {sid} — {col}: mean={subset[col].mean():.2f}, '
              f'std={subset[col].std():.2f}, '
              f'min={subset[col].min():.2f}, max={subset[col].max():.2f}')

sensor_dropdown.observe(update_raw_plot, names='value')
metric_dropdown.observe(update_raw_plot, names='value')

display(widgets.HBox([sensor_dropdown, metric_dropdown]))
display(out_raw)
update_raw_plot(None)

Output()

## 4. Dashboard — Actual vs Predicted

In [10]:
CONFIG_FE = {
    'lag_steps': [1, 2, 3],
    'lag_columns': ['flow', 'speed', 'occupancy'],
    'rolling_windows': [3, 6, 12],
    'rolling_columns': ['flow', 'speed'],
}
FEATURE_COLS = fe.get_feature_columns(CONFIG_FE)
TARGET_COL = 'flow_target'

baseline_dropdown = widgets.Dropdown(
    options=['Baseline_A', 'Baseline_B'],
    value='Baseline_A',
    description='Baseline:',
)

model_names = list(saved_models.get('Baseline_A', {}).keys()) or ['No models loaded']
model_dropdown = widgets.Dropdown(
    options=model_names,
    value=model_names[0] if model_names else None,
    description='Model:',
)

n_points_slider = widgets.IntSlider(
    value=300, min=50, max=3000, step=10,
    description='N points:',
)

out_pred = widgets.Output()

def update_prediction_plot(change):
    with out_pred:
        clear_output(wait=True)
        bl = baseline_dropdown.value
        mdl_name = model_dropdown.value
        n_pts = n_points_slider.value
        
        # Load test data
        test_key = f'baseline{bl[-1]}_test'
        if test_key not in data:
            print(f'❌ Không tìm thấy dữ liệu test cho {bl}')
            return
        
        df_test = data[test_key]
        if mdl_name not in saved_models.get(bl, {}):
            print(f'❌ Không tìm thấy model {mdl_name} cho {bl}')
            return
        
        model = saved_models[bl][mdl_name]
        X_test = df_test[FEATURE_COLS].values
        y_test = df_test[TARGET_COL].values
        y_pred = model.predict(X_test)
        
        # Tính metrics
        met = models.evaluate(y_test, y_pred)
        
        # Vẽ
        n = min(n_pts, len(y_test))
        fig, ax = plt.subplots(figsize=(16, 5))
        ax.plot(range(n), y_test[:n], label='Actual', alpha=0.8, linewidth=1)
        ax.plot(range(n), y_pred[:n], label='Predicted', alpha=0.8, linewidth=1)
        ax.set_title(f'{bl} — {mdl_name}', fontsize=14)
        ax.set_ylabel('Flow')
        ax.legend()
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()
        
        # Metrics + Traffic Alert
        print(f'📊 Metrics: MAE={met["MAE"]:.2f}  RMSE={met["RMSE"]:.2f}  '
              f'MAPE={met["MAPE"]:.2f}%  R²={met["R2"]:.4f}')
        
        avg_flow = np.mean(y_pred[:n])
        if avg_flow > 120:
            print(f'🔴 CẢNH BÁO: Tải cao bất thường — Cần giám sát ngay! (Avg Flow: {avg_flow:.1f})')
        elif avg_flow > 80:
            print(f'🟡 CẦN THEO DÕI: Lưu lượng tăng cao (Avg Flow: {avg_flow:.1f})')
        else:
            print(f'🟢 BÌNH THƯỜNG: Giao thông ổn định (Avg Flow: {avg_flow:.1f})')

baseline_dropdown.observe(update_prediction_plot, names='value')
model_dropdown.observe(update_prediction_plot, names='value')
n_points_slider.observe(update_prediction_plot, names='value')

display(widgets.HBox([baseline_dropdown, model_dropdown, n_points_slider]))
display(out_pred)
update_prediction_plot(None)

Output()

## 5. So sánh song song Baseline A vs B

In [6]:
# So sánh nhanh tất cả mô hình giữa 2 baselines
compare_model = widgets.Dropdown(
    options=model_names,
    value=model_names[0] if model_names else None,
    description='Model:',
)
out_compare = widgets.Output()

def update_compare(change):
    with out_compare:
        clear_output(wait=True)
        mdl_name = compare_model.value
        
        fig, axes = plt.subplots(1, 2, figsize=(18, 5))
        
        for ax, bl, color in zip(axes, ['Baseline_A', 'Baseline_B'], 
                                  ['#4C72B0', '#DD8452']):
            test_key = f'baseline{bl[-1]}_test'
            if test_key not in data or mdl_name not in saved_models.get(bl, {}):
                ax.set_title(f'{bl} — Không có dữ liệu')
                continue
            
            model = saved_models[bl][mdl_name]
            df_test = data[test_key]
            X_test = df_test[FEATURE_COLS].values
            y_test = df_test[TARGET_COL].values
            y_pred = model.predict(X_test)
            met = models.evaluate(y_test, y_pred)
            
            n = min(300, len(y_test))
            ax.plot(range(n), y_test[:n], label='Actual', alpha=0.8)
            ax.plot(range(n), y_pred[:n], label='Predicted', alpha=0.8, color=color)
            ax.set_title(f'{bl} — RMSE={met["RMSE"]:.2f}, R²={met["R2"]:.4f}')
            ax.legend()
            ax.grid(True, alpha=0.3)
        
        plt.suptitle(f'So sánh: {mdl_name}', fontsize=14)
        plt.tight_layout()
        plt.show()

compare_model.observe(update_compare, names='value')
display(compare_model)
display(out_compare)
update_compare(None)

Dropdown(description='Model:', options=('1_LinearRegression', '2_Ridge', '3_KNN', '4_DecisionTree', '5_RandomF…

Output()

---
**Dashboard hoàn tất.** Sử dụng các dropdown ở trên để khám phá kết quả.